# **IMPORT & LOAD DỮ LIỆU**

In [ ]:
# [3.1] Thu thập và nạp dữ liệu
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc

from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score

path = "/content/drive/MyDrive/DoAn_CD2_KHDL/"

train = reduce_mem_usage(pd.read_csv(path + 'train.csv'))
songs = reduce_mem_usage(pd.read_csv(path + 'songs.csv'))

In [ ]:
def reduce_mem_usage(df):
    for col in df.columns:
        if df[col].dtype == 'int64':
            df[col] = df[col].astype(np.int32)
        elif df[col].dtype == 'float64':
            df[col] = df[col].astype(np.float32)
    return df


In [ ]:

# [3.2.1] Phân bố hành vi nghe lại


print("✔ Load dữ liệu thành công")
print("Train shape:", train.shape)
print("Songs shape:", songs.shape)

display(train.head(3))
print("Target distribution:")
print(train['target'].value_counts(normalize=True))


# **KHÁM PHÁ DỮ LIỆU (EDA)**

In [ ]:
# [3.2.2] Phân tích hành vi theo nguồn phát (Source Type)
plt.figure(figsize=(12,5))
sns.barplot(x='source_type', y='target', data=train)
plt.title('Replay Probability by Source Type')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# [3.2.3] Phân tích hành vi theo ngôn ngữ bài hát
train_lang = train.merge(
    songs[['song_id','language']],
    on='song_id', how='left'
)

train_lang.groupby('language')['target'].mean() \
    .sort_values(ascending=False) \
    .head(10) \
    .plot(kind='bar', figsize=(12,5),
          title='Replay Probability by Language')
plt.show()


In [ ]:
# [3.2.4] Phân tích UI × Source Type
pivot = train.pivot_table(
    index='source_system_tab',
    columns='source_type',
    values='target',
    aggfunc='mean'
)

plt.figure(figsize=(12,6))
sns.heatmap(pivot, cmap='YlGnBu')
plt.title('UI × Source Type Interaction')
plt.show()


In [ ]:
# [3.3] Phân tích độ thưa dữ liệu
n_users = train['msno'].nunique()
n_items = train['song_id'].nunique()
sparsity = 1 - len(train)/(n_users*n_items)

print(f"Users: {n_users}")
print(f"Songs: {n_items}")
print(f"Sparsity: {sparsity:.4%}")


# **TIỀN XỬ LÝ DỮ LIỆU**

In [ ]:
# [3.4.1] Đặc trưng hành vi người dùng & bài hát (Bayesian smoothing)
global_mean = train['target'].mean()
print("Global repeat rate:", global_mean)

user_stats = train.groupby('msno')['target'].agg(['mean','count'])
user_stats['user_repeat_rate'] = (
    user_stats['mean'] * user_stats['count'] +
    global_mean * 20
) / (user_stats['count'] + 20)

user_repeat = user_stats['user_repeat_rate']
display(user_repeat.sample(5))



In [ ]:
song_stats = train.groupby('song_id')['target'].agg(['mean','count'])
song_stats['song_repeatability'] = (
    song_stats['mean'] * song_stats['count'] +
    global_mean * 30
) / (song_stats['count'] + 30)

song_repeat = song_stats['song_repeatability']
display(song_repeat.sample(5))


In [ ]:
# [3.4.2] Lọc nhiễu và mã hóa người dùng – bài hát
valid_users = train['msno'].value_counts()[lambda x: x >= 20].index
valid_songs = train['song_id'].value_counts()[lambda x: x >= 15].index

train_f = train[
    train['msno'].isin(valid_users) &
    train['song_id'].isin(valid_songs)
].copy()

train_f['u_code'] = train_f['msno'].astype('category').cat.codes
train_f['s_code'] = train_f['song_id'].astype('category').cat.codes

print("Sau lọc nhiễu:")
print("Users:", train_f['msno'].nunique())
print("Songs:", train_f['song_id'].nunique())
print("Interactions:", len(train_f))


In [ ]:
del train
gc.collect()


# **Train mô hình**

In [ ]:
# [4.1] Chia dữ liệu train history / future (Pseudo time split)
train_f = train_f.sort_values(['msno','song_id'])
train_f['order'] = train_f.groupby('msno').cumcount()

cut = train_f.groupby('msno')['order'].transform(
    lambda x: np.quantile(x, 0.8)
)

train_hist   = train_f[train_f['order'] <= cut].copy()
train_future = train_f[train_f['order'] > cut].copy()

print("Train history:", train_hist.shape)
print("Train future:", train_future.shape)

In [ ]:
# Hệ 1
user_norm = train_hist.groupby('u_code')['target'].sum()
# Add a small epsilon to user_norm to prevent division by zero
user_norm_safe = user_norm.replace(0, 1e-6) # Replace 0 with a small number to avoid inf

item_pop  = np.log1p(train_hist.groupby('s_code')['target'].sum())

train_hist['cf_weight'] = (
    train_hist['target'] /
    train_hist['u_code'].map(user_norm_safe) # Use the safe version here
) / (1 + train_hist['s_code'].map(item_pop))

In [ ]:
#Hệ1
u_i = csr_matrix(
    (train_hist['cf_weight'],
     (train_hist['u_code'], train_hist['s_code']))
)

print("User-item matrix shape:", u_i.shape)


In [ ]:
#Hệ 1
MAX_USER_SIM = 5000
user_sim = cosine_similarity(u_i[:MAX_USER_SIM])
print("User similarity matrix:", user_sim.shape)


In [ ]:
def discovery_rec(u_code, n=10, top_k=50):
    sims = user_sim[u_code]
    neighbors = sims.argsort()[-(top_k+1):-1][::-1]
    scores = u_i[neighbors].T.dot(sims[neighbors])
    scores[u_i[u_code].indices] = -1
    return scores.argsort()[-n:][::-1]


In [ ]:
# [4.3] Hệ  khuyến nghị nghe lại (Repeat Listening)
replay = (
    train_hist
    .groupby(['msno','song_id'])
    .agg(
        play_count=('target','count'),
        repeat_rate=('target','mean'),
        library_hits=('source_type',
                      lambda x: (x=='local-library').sum()),
        last_order=('order','max')
    )
    .reset_index()
)

print("Replay interaction shape:", replay.shape)
display(replay.head())


In [ ]:
replay = replay.merge(user_repeat, on='msno', how='left')
replay = replay.merge(song_repeat, on='song_id', how='left')


In [ ]:
replay['play_norm'] = np.log1p(replay['play_count'])
replay['library_flag'] = (replay['library_hits'] > 0).astype(int)
replay['recency_decay'] = np.exp(-replay['last_order'] / 6)


In [ ]:
replay['replay_score'] = (
    0.28 * replay['repeat_rate'] +
    0.24 * replay['user_repeat_rate'] +
    0.20 * replay['song_repeatability'] +
    0.16 * replay['play_norm'] +
    0.12 * replay['library_flag']
) * replay['recency_decay']

replay_final = replay.sort_values('replay_score', ascending=False)
display(replay_final.head(10))


# **ĐÁNH GIÁ MÔ HÌNH**

In [ ]:
# [4.5] Đánh giá mô hình
y_true = (replay_final['repeat_rate'] > 0.5).astype(int)
y_score = replay_final['replay_score']

auc = roc_auc_score(y_true, y_score)
print(f"Replay ROC-AUC: {auc:.4f}")


In [ ]:
# [4.5] Đánh giá mô hình
def precision_at_k(df, k=10):
    res = []
    for _, g in df.groupby('msno'):
        if len(g) >= k:
            topk = g.sort_values(
                'replay_score', ascending=False
            ).head(k)
            res.append((topk['repeat_rate'] > 0.5).mean())
    return np.mean(res)

print("Precision@10:", precision_at_k(replay_final))


In [ ]:
# [4.5] Đánh giá mô hình
coverage = replay_final[
    replay_final['replay_score'] > 0.6
]['song_id'].nunique()

print(
    f"Catalog coverage: "
    f"{coverage / replay_final['song_id'].nunique():.2%}"
)


# **TRIỂN KHAI & DEMO KẾT QUẢ**

In [ ]:
# [4.4] Hybrid Recommendation
s_map = train_f[['s_code','song_id']].drop_duplicates().set_index('s_code')

def hybrid_recommend(user_id, n=10):

    if user_id not in user_repeat.index:
        print("❌ User không tồn tại")
        return None

    # ---- REPEAT SYSTEM
    if user_repeat.loc[user_id] > 0.65:
        rec = replay_final[
            replay_final['msno'] == user_id
        ].head(n)[['song_id','replay_score']]

        if len(rec) == 0:
            print("⚠ User có repeat rate cao nhưng không có replay history")
        return rec

    # ---- DISCOVERY SYSTEM
    else:
        rows = train_f[train_f['msno'] == user_id]
        if len(rows) == 0:
            print("❌ User không có interaction trong train_f")
            return None

        u_code = rows['u_code'].iloc[0]
        s_codes = discovery_rec(u_code, n=n)

        return s_map.loc[s_codes].reset_index(drop=True)



In [ ]:
# Find a sample user whose u_code is within the MAX_USER_SIM limit for discovery_rec
valid_u_codes_for_discovery = train_f[train_f['u_code'] < MAX_USER_SIM]['msno'].unique()
if len(valid_u_codes_for_discovery) > 0:
    sample_user_for_discovery = valid_u_codes_for_discovery[0] # Pick the first valid user
else:
    # Fallback if no user is found within the limit (unlikely with MAX_USER_SIM=5000 and 24k users)
    sample_user_for_discovery = replay_final['msno'].iloc[0] # Original sample user, will still error for discovery_rec

# Use the first user in replay_final for repeat-listening rec, as it doesn't have the MAX_USER_SIM restriction
sample_user_for_repeat = replay_final['msno'].iloc[0]

print("🔁 REPEAT-LISTENING RECOMMENDATION")
display(hybrid_recommend(sample_user_for_repeat, n=10))

print("🔍 DISCOVERY RECOMMENDATION")
u_code_for_discovery = train_f.loc[train_f['msno']==sample_user_for_discovery,'u_code'].iloc[0]
if u_code_for_discovery < MAX_USER_SIM:
    print(discovery_rec(u_code_for_discovery, n=10))
else:
    print(f"User with u_code {u_code_for_discovery} is outside the {MAX_USER_SIM} user limit for discovery recommendation. Cannot provide discovery recommendations for this user.")

print("\n✅ PIPELINE 6 BƯỚC – CHẠY HOÀN TẤT")